<a href="https://colab.research.google.com/github/i-wild/Python_Ejercicios/blob/master/v09_engine_cfdi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PRIMERA VERSIÓN FUNCIONAL

In [ ]:
# ============================================================
# KPI ENGINE V8.3 FINAL 🔥
# COMPLETO + ROBUSTO + UX + SIN ERRORES SAT
# ============================================================

import pandas as pd
from google.colab import files
from IPython.display import Audio, display
import numpy as np

# ============================================================
# 🔔 SONIDO
# ============================================================

def beep():
    duration = 0.2
    freq = 440
    sample_rate = 44100
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    tone = np.sin(freq * t * 2 * np.pi)
    display(Audio(tone, rate=sample_rate, autoplay=True))

# ============================================================
# 1. CARGA DINÁMICA DE ARCHIVOS 🔥
# ============================================================

archivos = []

while True:
    print("\n📂 Sube uno o varios archivos...")

    uploaded = files.upload()
    nuevos = list(uploaded.keys())

    archivos.extend(nuevos)

    beep()

    print(f"✅ Se agregaron {len(nuevos)} archivos")
    print(f"📊 Total acumulado: {len(archivos)} archivos")

    continuar = input("¿Deseas agregar más archivos? (s/n): ").lower()

    if continuar != "s":
        break

# ============================================================
# 2. ELIMINAR ARCHIVOS (OPCIONAL)
# ============================================================

print("\n📋 Archivos cargados:")
for i, f in enumerate(archivos):
    print(f"{i}: {f}")

while True:
    eliminar = input("\n¿Deseas eliminar algún archivo? (s/n): ").lower()
    if eliminar != "s":
        break

    idx = int(input("Índice a eliminar: "))

    if 0 <= idx < len(archivos):
        eliminado = archivos.pop(idx)
        print(f"❌ Eliminado: {eliminado}")
    else:
        print("⚠️ Índice inválido")

# ============================================================
# 3. CONFIRMACIÓN FINAL
# ============================================================

confirmar = input("\n¿Deseas iniciar el análisis? (s/n): ").lower()

if confirmar != "s":
    print("⛔ Proceso cancelado")
    raise SystemExit

print("\n🚀 Iniciando procesamiento...")

# ============================================================
# 4. PROCESAMIENTO ROBUSTO 🔥
# ============================================================

dataframes = []

for file_name in archivos:
    print(f"\n⚙️ Procesando: {file_name}")

    with open(file_name, 'r', encoding='latin-1') as f:
        first_line = next(f)

    if "~" in first_line:
        sep = "~"
    elif "|" in first_line:
        sep = "|"
    else:
        sep = ","

    try:
        df_temp = pd.read_csv(
            file_name,
            sep=sep,
            encoding='latin-1',
            engine='python',
            on_bad_lines='skip'
        )

        print(f"✅ OK: {df_temp.shape[0]} filas")

    except Exception as e:
        print(f"❌ Error en {file_name}: {e}")
        continue

    dataframes.append(df_temp)

df = pd.concat(dataframes, ignore_index=True)

del dataframes
del archivos

print("\n📐 Total registros:", df.shape)

# ============================================================
# 5. NORMALIZACIÓN
# ============================================================

df.columns = df.columns.str.strip()

def find_col(keywords):
    for col in df.columns:
        for k in keywords:
            if k.lower() in col.lower():
                return col
    return None

CONFIG = {
    "RFC": find_col(["RfcReceptor"]),
    "FECHA": find_col(["FechaEmision"]),
    "TOTAL": find_col(["Monto"]),
    "UUID": find_col(["Uuid"])
}

print("🧠 Column mapping:", CONFIG)

# ============================================================
# 6. LIMPIEZA
# ============================================================

df[CONFIG["FECHA"]] = pd.to_datetime(df[CONFIG["FECHA"]], errors='coerce')
df[CONFIG["TOTAL"]] = pd.to_numeric(df[CONFIG["TOTAL"]], errors='coerce')

df.dropna(subset=[CONFIG["FECHA"], CONFIG["TOTAL"]], inplace=True)
df.drop_duplicates(subset=[CONFIG["UUID"]], inplace=True)

# ============================================================
# 7. OPTIMIZACIÓN
# ============================================================

df[CONFIG["RFC"]] = df[CONFIG["RFC"]].astype("category")
df[CONFIG["UUID"]] = df[CONFIG["UUID"]].astype("category")
df[CONFIG["TOTAL"]] = pd.to_numeric(df[CONFIG["TOTAL"]], downcast="float")

# ============================================================
# 8. VARIABLES TIEMPO
# ============================================================

df['Mes'] = df[CONFIG["FECHA"]].dt.to_period('M')
df['Fecha_dia'] = pd.to_datetime(df[CONFIG["FECHA"]].dt.date)

# ============================================================
# 9. CLASIFICACIÓN RFC
# ============================================================

def clasificar_rfc(rfc):
    if pd.isna(rfc):
        return "Desconocido"
    rfc = str(rfc).strip().upper()
    if rfc == "XAXX010101000":
        return "Genérico Nacional"
    if rfc == "XEXX010101000":
        return "Genérico Extranjero"
    if len(rfc) == 12:
        return "Persona Moral"
    elif len(rfc) == 13:
        return "Persona Física"
    return "Otro"

df['Tipo_Persona'] = df[CONFIG["RFC"]].apply(clasificar_rfc)

# ============================================================
# 10. KPI CONSOLA 🔥 (NO SE VUELVE A PERDER)
# ============================================================

print("\n================= RESUMEN KPI =================")

total_cfdi = len(df)
print(f"📦 Total CFDI: {total_cfdi:,}")

print("\n--- CFDI POR TIPO ---")
for tipo in df['Tipo_Persona'].unique():
    count = (df['Tipo_Persona'] == tipo).sum()
    print(f"{tipo}: {count:,}")

print("\n--- TOP 10 PROVEEDORES ---")
print(df[CONFIG["RFC"]].value_counts().head(10))

# ============================================================
# 11. KPI GENERAL
# ============================================================

resumen = pd.DataFrame({
    "Total CFDI": [total_cfdi],
    "Ingresos": [df[CONFIG["TOTAL"]].sum()],
    "Ticket Promedio": [df[CONFIG["TOTAL"]].mean()],
    "Clientes Únicos": [df[CONFIG["RFC"]].nunique()]
})

# ============================================================
# 12. CFDI POR DÍA
# ============================================================

cfdi_emitidos = pd.crosstab(df['Fecha_dia'], df['Tipo_Persona'])

# ============================================================
# 13. MENSUAL
# ============================================================

mensual = df.groupby('Mes')[CONFIG["UUID"]].count().reset_index()
mensual.columns = ['Mes', 'CFDI']
mensual['Variacion_%'] = mensual['CFDI'].pct_change() * 100

# ============================================================
# 14. PROVEEDORES
# ============================================================

proveedores = df.groupby(CONFIG["RFC"]).agg(
    CFDI_Total=(CONFIG["UUID"], "count"),
    Monto_Total=(CONFIG["TOTAL"], "sum"),
    Tipo=("Tipo_Persona", "first")
).reset_index()

proveedores_mensual = df.groupby(
    [CONFIG["RFC"], "Mes"]
)[CONFIG["UUID"]].count().reset_index()

# ============================================================
# 15. CORTES DINÁMICOS
# ============================================================

cfdi_dia = df.groupby('Fecha_dia')[CONFIG["UUID"]].count().sort_index()

LIMITE = 1_000_000
UMBRAL = LIMITE * 0.95

cortes = []
inicio = None
acum = 0

for fecha, val in cfdi_dia.items():

    if inicio is None:
        inicio = fecha

    if acum + val > UMBRAL:
        cortes.append([inicio, fecha_prev, acum])
        inicio = fecha
        acum = val
    else:
        acum += val

    fecha_prev = fecha

cortes.append([inicio, fecha_prev, acum])

cortes_df = pd.DataFrame(cortes, columns=["Inicio", "Fin", "CFDI"])

# ============================================================
# 16. EXPORTACIÓN
# ============================================================

archivo_salida = "KPI_SAT_V8_3_FINAL.xlsx"

with pd.ExcelWriter(archivo_salida) as writer:
    resumen.to_excel(writer, sheet_name="Resumen", index=False)
    cfdi_emitidos.to_excel(writer, sheet_name="CFDI_EMITIDOS")
    mensual.to_excel(writer, sheet_name="Mensual", index=False)
    proveedores.to_excel(writer, sheet_name="PROVEEDORES", index=False)
    proveedores_mensual.to_excel(writer, sheet_name="PROV_MENSUAL", index=False)
    cortes_df.to_excel(writer, sheet_name="CORTES", index=False)

print("\n✅ Archivo generado:", archivo_salida)
files.download(archivo_salida)


📂 Sube uno o varios archivos...


Saving 1E4122DD-A54B-4BA2-AF32-6CC5E01EEB93_01.txt to 1E4122DD-A54B-4BA2-AF32-6CC5E01EEB93_01 (1).txt
Saving 2BF560B3-8DC0-4417-96FB-0D88EA984385_01.txt to 2BF560B3-8DC0-4417-96FB-0D88EA984385_01 (1).txt
Saving 2D2A361C-1DBD-4DC2-94CA-583D3CB2E0D9_01.txt to 2D2A361C-1DBD-4DC2-94CA-583D3CB2E0D9_01 (1).txt
Saving 2E4E14BB-2C62-4AA6-866B-2B42F260C53A_01.txt to 2E4E14BB-2C62-4AA6-866B-2B42F260C53A_01 (1).txt
Saving 4A468204-FCB0-4666-AADB-6C5FE872EC4B_01.txt to 4A468204-FCB0-4666-AADB-6C5FE872EC4B_01 (1).txt
Saving 4A468204-FCB0-4666-AADB-6C5FE872EC4B_tercero.txt to 4A468204-FCB0-4666-AADB-6C5FE872EC4B_tercero (1).txt
Saving 05FAF6DC-DD27-4900-B111-FA453243869A_01.txt to 05FAF6DC-DD27-4900-B111-FA453243869A_01 (1).txt
Saving 05FAF6DC-DD27-4900-B111-FA453243869A_tercero.txt to 05FAF6DC-DD27-4900-B111-FA453243869A_tercero (1).txt
Saving 8B7F07BB-F5D1-442D-B015-3C9EF8582A76_01.txt to 8B7F07BB-F5D1-442D-B015-3C9EF8582A76_01 (1).txt
Saving 9BF366BA-13FC-49A5-9188-CE4272D00318_01.txt to 9BF366BA

✅ Se agregaron 46 archivos
📊 Total acumulado: 46 archivos
¿Deseas agregar más archivos? (s/n): n

📋 Archivos cargados:
0: 1E4122DD-A54B-4BA2-AF32-6CC5E01EEB93_01 (1).txt
1: 2BF560B3-8DC0-4417-96FB-0D88EA984385_01 (1).txt
2: 2D2A361C-1DBD-4DC2-94CA-583D3CB2E0D9_01 (1).txt
3: 2E4E14BB-2C62-4AA6-866B-2B42F260C53A_01 (1).txt
4: 4A468204-FCB0-4666-AADB-6C5FE872EC4B_01 (1).txt
5: 4A468204-FCB0-4666-AADB-6C5FE872EC4B_tercero (1).txt
6: 05FAF6DC-DD27-4900-B111-FA453243869A_01 (1).txt
7: 05FAF6DC-DD27-4900-B111-FA453243869A_tercero (1).txt
8: 8B7F07BB-F5D1-442D-B015-3C9EF8582A76_01 (1).txt
9: 9BF366BA-13FC-49A5-9188-CE4272D00318_01 (1).txt
10: 9BF366BA-13FC-49A5-9188-CE4272D00318_tercero (1).txt
11: 32E4491D-E7C1-4C24-9567-79144B188FED_01 (1).txt
12: 63F68605-328A-4A6D-942D-B461C67BBE25_01 (1).txt
13: 63F68605-328A-4A6D-942D-B461C67BBE25_tercero (1).txt
14: 72AEA293-1E2A-4EFC-ABD0-2F745A321372_01 (1).txt
15: 72AEA293-1E2A-4EFC-ABD0-2F745A321372_tercero (1).txt
16: 89F07DD1-6DD6-4641-8340-D12EA0

/tmp/ipykernel_3702/3014812433.py:236: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  proveedores = df.groupby(CONFIG["RFC"]).agg(
/tmp/ipykernel_3702/3014812433.py:242: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  proveedores_mensual = df.groupby(



✅ Archivo generado: KPI_SAT_V8_3_FINAL.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>